# Download FineWeb-Edu Raw Data

Downloads ~2B tokens worth of raw text from FineWeb-Edu.
Saves raw text so you can tokenize with different tokenizers later.

**Output:** Raw text dataset with train/val/test splits

In [1]:
from pathlib import Path
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer
from tqdm.auto import tqdm
import random

# Configuration
OUTPUT_DIR = Path("../data/fineweb-edu-raw")
TARGET_TOKENS = 2_000_000_000  # ~2B tokens (estimated)
SEED = 42

# We use a tokenizer just to estimate token count, not to save tokens
ESTIMATOR_TOKENIZER = "gpt2"

/Users/ahmetcanyavuz/Developer/lm-trainer/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## 1. Stream and Download Raw Text

In [2]:
# Load tokenizer just for estimating token counts
tokenizer = AutoTokenizer.from_pretrained(ESTIMATOR_TOKENIZER)
print(f"Using {ESTIMATOR_TOKENIZER} to estimate token counts")

Using gpt2 to estimate token counts


In [3]:
# Stream FineWeb-Edu
dataset = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    split="train",
    streaming=True,
)

print("Streaming dataset loaded")

Streaming dataset loaded


In [4]:
# Collect raw text documents until we reach target tokens
documents = []
total_tokens = 0

pbar = tqdm(total=TARGET_TOKENS, unit="tok", desc="Collecting data")

for i, example in enumerate(dataset):
    text = example["text"]
    
    # Estimate token count (for progress tracking)
    est_tokens = len(tokenizer.encode(text, add_special_tokens=False))
    
    # Skip very short documents
    if est_tokens < 50:
        continue
    
    # Save RAW TEXT, not tokens
    documents.append({
        "text": text,
        "uid": len(documents),
    })
    
    total_tokens += est_tokens
    pbar.update(est_tokens)
    
    if total_tokens >= TARGET_TOKENS:
        break
    
    if len(documents) % 10000 == 0:
        pbar.set_postfix({"docs": len(documents)})

pbar.close()

print(f"\nCollected {len(documents):,} documents")
print(f"Estimated tokens: {total_tokens:,} ({total_tokens/1e9:.2f}B)")


Collected 1,895,346 documents
Estimated tokens: 2,000,000,611 (2.00B)


## 2. Create Train/Val/Test Splits

In [5]:
# Shuffle
random.seed(SEED)
random.shuffle(documents)

# Split: 95% train, 2.5% val, 2.5% test
n = len(documents)
train_end = int(0.95 * n)
val_end = int(0.975 * n)

train_docs = documents[:train_end]
val_docs = documents[train_end:val_end]
test_docs = documents[val_end:]

print(f"Train: {len(train_docs):,} documents")
print(f"Val:   {len(val_docs):,} documents")
print(f"Test:  {len(test_docs):,} documents")

Train: 1,800,578 documents
Val:   47,384 documents
Test:  47,384 documents


## 3. Save Raw Text Data

In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_dataset = Dataset.from_list(train_docs)
val_dataset = Dataset.from_list(val_docs)
test_dataset = Dataset.from_list(test_docs)

train_dataset.save_to_disk(OUTPUT_DIR / "train")
val_dataset.save_to_disk(OUTPUT_DIR / "val")
test_dataset.save_to_disk(OUTPUT_DIR / "test")

print(f"Saved raw text to {OUTPUT_DIR}")

Saving the dataset (1/1 shards): 100%|██████████| 47384/47384 [00:00<00:00, 829014.24 examples/s]

Saved raw text to ../data/fineweb-edu-raw


## 4. Verify

In [ ]:
from datasets import load_from_disk

check = load_from_disk(str(OUTPUT_DIR / "train"))
print(f"Columns: {check.column_names}")
print(f"\nFirst document preview:")
print(check[0]["text"][:500] + "...")

Columns: ['text', 'uid']

First document preview:
London's Natural History (Facsimile Edition)
This facsimile reprint is identical in every way to the original first edition.
Richard Fitter provided the first comprehensive history of a great human community in terms of the animals and plants it has displaced, changed, moved and removed, introduced, conserved, lost or forgotten. In selecting London as an area for such study, Mr.Fitter, himself a Londoner, took the world's largest aggregation of human beings living in a single community and in ma...


: 

## Done!

Raw text saved to `data/fineweb-edu-raw/`

Now use notebook `02_tokenize_data.ipynb` to tokenize with any tokenizer.